<a href="https://colab.research.google.com/github/nayanjha16/CodeGen-Implementations-May_26/blob/Group-30/app1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
  # ============================================================
  #  CodeGen-30 - Text or Voice input UI (Gradio + Whisper ASR)
  # ============================================================
  !pip install -q gradio

  import torch
  import gradio as gr
  from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

  DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

  # ---- 1. ASR: voice -> text. whisper-base is small & fast. ----
  _asr = pipeline(
      "automatic-speech-recognition",
      model="openai/whisper-base",
      device=0 if DEVICE == "cuda" else -1,
  )

  # ---- 2. Code generator: reuse the notebook's model if present,
  #         else load the base codegen model. ----
  def _ensure_codegen():
      global tokenizer_codegen, model_codegen
      if "model_codegen" in globals() and "tokenizer_codegen" in globals():
          return tokenizer_codegen, model_codegen
      name = "Salesforce/codegen-350M-multi"
      tok = AutoTokenizer.from_pretrained(name)
      if tok.pad_token_id is None:
          tok.pad_token_id = tok.eos_token_id
      if DEVICE == "cuda":
          mdl = AutoModelForCausalLM.from_pretrained(
              name, torch_dtype=torch.float16, device_map="auto")
      else:
          mdl = AutoModelForCausalLM.from_pretrained(name).to(DEVICE)
      tokenizer_codegen, model_codegen = tok, mdl
      return tok, mdl

  def generate_code(intent_text, target_lang):
      tok, mdl = _ensure_codegen()
      # To use the fine-tuned model later: swap `mdl` for `model_codegen_lora`.
      prompt = (f"Generate {target_lang} code based on the following "
                f"documentation:\n{intent_text}\n{target_lang} code:\n")
      inputs = tok(prompt, return_tensors="pt").to(mdl.device)
      out = mdl.generate(**inputs, max_new_tokens=256, do_sample=True,
                         top_k=50, pad_token_id=tok.eos_token_id)
      text = tok.decode(out[0], skip_special_tokens=True)
      marker = f"{target_lang} code:"
      return text.split(marker, 1)[1].strip() if marker in text else text.strip()

  # ---- 3. Single entry point: handles BOTH text and voice ----
  def handle(typed_text, mic_audio, target_lang):
      intent = (typed_text or "").strip()
      if mic_audio is not None:                      # voice path -> transcribe
          intent = _asr(mic_audio)["text"].strip()
      if not intent:
          return "Type an intent or record audio first.", ""
      return intent, generate_code(intent, target_lang)

  # ---- 4. Gradio UI ----
  demo = gr.Interface(
      fn=handle,
      inputs=[
          gr.Textbox(label="Type your intent (what should the code do?)", lines=3),
          gr.Audio(sources=["microphone"], type="filepath", label="...or speak it"),
          gr.Dropdown(["python", "cpp"], value="python", label="Target language"),
      ],
      marker = f"{target_lang} code:"
      return text.split(marker, 1)[1].strip() if marker in text else text.strip()

  # ---- 3. Single entry point: handles BOTH text and voice ----
  def handle(typed_text, mic_audio, target_lang):
      intent = (typed_text or "").strip()
      if mic_audio is not None:                      # voice path -> transcribe
          intent = _asr(mic_audio)["text"].strip()
      if not intent:
          return "Type an intent or record audio first.", ""
      return intent, generate_code(intent, target_lang)

  # ---- 4. Gradio UI ----
  demo = gr.Interface(
      fn=handle,
      inputs=[
          gr.Textbox(label="Type your intent (what should the code do?)", lines=3),
      tok, mdl = _ensure_codegen()
      # To use the fine-tuned model later: swap `mdl` for `model_codegen_lora`.
      prompt = (f"Generate {target_lang} code based on the following "
                f"documentation:\n{intent_text}\n{target_lang} code:\n")
      inputs = tok(prompt, return_tensors="pt").to(mdl.device)
      out = mdl.generate(**inputs, max_new_tokens=256, do_sample=True,
                         top_k=50, pad_token_id=tok.eos_token_id)
      text = tok.decode(out[0], skip_special_tokens=True)
      marker = f"{target_lang} code:"
      return text.split(marker, 1)[1].strip() if marker in text else text.strip()

  # ---- 3. Single entry point: handles BOTH text and voice ----
  def handle(typed_text, mic_audio, target_lang):
      intent = (typed_text or "").strip()
      if mic_audio is not None:                      # voice path -> transcribe
          intent = _asr(mic_audio)["text"].strip()
      if not intent:
          return "Type an intent or record audio first.", ""
      return intent, generate_code(intent, target_lang)

  # ---- 4. Gradio UI ----
  demo = gr.Interface(
      fn=handle,
      inputs=[
          gr.Textbox(label="Type your intent (what should the code do?)", lines=3),
          gr.Audio(sources=["microphone"], type="filepath", label="...or speak it"),
          gr.Dropdown(["python", "cpp"], value="python", label="Target language"),
      ],
      outputs=[
          gr.Textbox(label="Understood intent (from text/voice)"),
          gr.Code(label="Generated code"),
      ],
      title="CodeGen-30 - text or voice to code",
      description="Type or speak what you want; Whisper turns speech into text, "
                  "then codegen generates the code.",
  )

  demo.launch(share=True)   # share=True -> public link, works in Colab